In [1]:
%%writefile requirements.txt
tensorflow>=2.13.0
scikit-learn==1.2.2
pyod==1.1.2
pandas>=2.0.0

Writing requirements.txt


In [6]:
%%writefile scheduler.py
import schedule
import time
import subprocess
import boto3
from datetime import datetime

#bucket and file
BUCKET_NAME = "sagemaker-us-east-1-197337164107" 
PREFIX = "raw/" 

def get_latest_s3_file(bucket, prefix):
    """Scans the S3 bucket and returns the s3:// path of the newest file."""
    s3_client = boto3.client('s3')
    
    try:
        response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
        if 'Contents' not in response:
            print("No files found in the specified S3 bucket/prefix.")
            return None
        # Sort the files
        sorted_files = sorted(response['Contents'], key=lambda x: x['LastModified'], reverse=True)
        # Get the file path
        latest_file_key = sorted_files[0]['Key']
        if latest_file_key.endswith('/'):
            return None   
        full_s3_path = f"s3://{bucket}/{latest_file_key}"
        return full_s3_path    
    except Exception as e:
        print(f"Error accessing S3: {e}")
        return None

def run_inference_pipeline():
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] Waking up for scheduled run...")
    latest_file = get_latest_s3_file(BUCKET_NAME, PREFIX)
    if not latest_file:
        print("Skipping run. No valid file found to process.")
        return
    print(f"Latest file found: {latest_file}")
    print("Triggering inference.py...")
    try:
        subprocess.run([
            "python", 
            "inference.py", 
            "--input-raw", 
            latest_file
        ], check=True)
        print("Job completed successfully. Dashboard is ready to read the new CSV.")
    except subprocess.CalledProcessError as e:
        print(f"Error running inference script: {e}")

# --- Scheduler ---
print(f"Starting pipeline scheduler. Monitoring s3://{BUCKET_NAME}/{PREFIX}")
print("Running initial job right now to kick things off...")
run_inference_pipeline() 
# Now this runs every 5 min
schedule.every(5).minutes.do(run_inference_pipeline)

while True:
    schedule.run_pending()
    time.sleep(1)

Writing scheduler.py


In [1]:
%%writefile serve.py
import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from pyod.models.gmm import GMM

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def model_fn(model_dir):
    print("Loading Keras AutoEncoder...")
    ae_path = os.path.join(model_dir, 'autoencoder_model.keras')
    autoencoder = tf.keras.models.load_model(ae_path)
    
    print("Loading Isolation Forest...")
    iso_path = os.path.join(model_dir, 'isolation_forest.pkl')
    isolation_forest = joblib.load(iso_path)
    
    print("Loading GMM...")
    gmm_path = os.path.join(model_dir, 'gmm_model.pkl')
    gmm_model = joblib.load(gmm_path)
    
    print("Loading Scaler...")
    scaler_path = os.path.join(model_dir, 'scaler.pkl')
    scaler = joblib.load(scaler_path)
    
    print("Loading Metadata and Thresholds...")
    metadata_path = os.path.join(model_dir, 'model_metadata.joblib')
    metadata = joblib.load(metadata_path)
    
    return {
        'autoencoder': autoencoder,
        'isolation_forest': isolation_forest,
        'gmm_model': gmm_model,
        'scaler': scaler,
        'metadata': metadata
    }

def input_fn(request_body, request_content_type):
    if request_content_type == 'application/json':
        data = json.loads(request_body)
        return pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}. Expected 'application/json'.")

def predict_fn(input_data, model_dict):
    autoencoder = model_dict['autoencoder']
    isolation_forest = model_dict['isolation_forest']
    gmm_model = model_dict['gmm_model']
    scaler = model_dict['scaler']
    metadata = model_dict['metadata']
    
    # 1. Extract Thresholds
    ae_thresh = metadata['ae_threshold']
    iso_thresh = metadata['iso_threshold']
    gmm_thresh = metadata['gmm_threshold']
    
    # 2. Scale Features
    features = input_data[metadata['feature_names']]
    X_scaled = scaler.transform(features).astype(np.float32)
    
    # 3. Generate Raw Scores
    reconstructed = autoencoder.predict(X_scaled, verbose=0)
    ae_mse = np.mean(np.square(X_scaled - reconstructed), axis=1)
    if_scores = isolation_forest.decision_function(X_scaled)
    gmm_scores = gmm_model.decision_function(X_scaled)
    
    # 4. 0-10 Scaling (Using safe fallbacks for single-row inference)
    ae_min = metadata.get('ae_min_error', 0.0) 
    ae_scale_factor = (ae_thresh - ae_min) if (ae_thresh - ae_min) > 0 else 1e-6
    ae_score_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    if_max = metadata.get('if_max_score', 0.15) 
    if_scale_factor = (if_max - iso_thresh) if (if_max - iso_thresh) > 0 else 1e-6
    if_score_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    # FIXED: Hardcoded safe minimum for GMM instead of np.min()
    gmm_min = metadata.get('gmm_min_score', -25.0) 
    gmm_scale_factor = (gmm_thresh - gmm_min) if (gmm_thresh - gmm_min) > 0 else 1e-6
    gmm_score_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    # 5. Ensemble Average
    ensemble_score_10 = float((ae_score_10[0] + if_score_10[0] + gmm_score_10[0]) / 3.0)
    
    # 6. JSON Return Payload
    return {
        "frustrationScore": round(ensemble_score_10, 2),
        "severity": calculate_severity(ensemble_score_10),
        "breakdown": {
            "ae_raw_mse": float(ae_mse[0]),
            "if_raw_score": float(if_scores[0]),
            "gmm_raw_score": float(gmm_scores[0])
        }
    }

def output_fn(prediction, accept):
    if accept == 'application/json':
        return json.dumps(prediction), accept
    else:
        raise ValueError(f"Accept header must be 'application/json', got: {accept}")

Overwriting serve.py


In [2]:
import os
import sagemaker
from sagemaker.sklearn.model import SKLearnModel
import boto3
import json
from sagemaker.serverless import ServerlessInferenceConfig

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
sess = sagemaker.Session()
role = sagemaker.get_execution_role() 

current_dir = os.path.abspath(os.getcwd())
req_path = os.path.join(current_dir, "requirements.txt")
serve_path = os.path.join(current_dir, "serve.py") 

model_s3_path = "s3://sagemaker-studio-i0gutcxdy/frustration-model/models/model.tar.gz"

print(f"Using requirements from: {req_path}")
print(f"Using entry point from: {serve_path}")
print("Configuring the Ensemble Model for SageMaker...")

ensemble_model = SKLearnModel(
    model_data=model_s3_path,
    role=role,
    entry_point=serve_path,
    dependencies=[req_path],
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sess
)
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5
)

print("Deploying Serverless endpoint)")
predictor = ensemble_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="frustration-serverless-endpoint"
)

print(f"\n[SUCCESS] Serverless Endpoint deployed at: {predictor.endpoint_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Using requirements from: /home/sagemaker-user/deployment/requirements.txt
Using entry point from: /home/sagemaker-user/deployment/serve.py
Configuring the Ensemble Model for SageMaker...
Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)
------------------------------------------

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:29                                                                                   │
│                                                                                                  │
│   26 )                                                                                           │
│   27                                                                                             │
│   28 print("Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)")                     │
│ ❱ 29 predictor = ensemble_model.deploy(                                                          │
│   30 │   initial_instance_count=1,                                                               │
│   31 │   instance_type="ml.m5.large",                                                            │
│   32 │   endpoint_name="frustration-ensemble-endpoint"                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model.py:1814 in deploy                        │
│                                                                                                  │
│   1811 │   │   │   │   )                                                                         │
│   1812 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1813 │   │   │   else:                                                                         │
│ ❱ 1814 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1815 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1816 │   │   │   │   │   production_variants=[production_variant],                             │
│   1817 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:6033 in                             │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6030 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6031 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6032 │   │                                                                                     │
│ ❱ 6033 │   │   return self.create_endpoint(                                                      │
│   6034 │   │   │   endpoint_name=name,                                                           │
│   6035 │   │   │   config_name=name,                                                             │
│   6036 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:4867 in create_endpoint             │
│                                                                                                  │
│   4864 │   │   │   │   self.endpoint_arn = res["EndpointArn"]                                    │
│   4865 │   │   │                                                                                 │
│   4866 │   │   │   if wait:                                                                      │
│ ❱ 4867 │   │   │   │   self.wait_for_endpoint(endpoint_name, live_logging=live_logging)          │
│   4868 │   │   │   return endpoint_name                                                          │
│   4869 │   │   except Exception as e:                      

In [ ]:
import boto3
import json

runtime_client = boto3.client('sagemaker-runtime')

test_payload = {
    "event_count": 45,
    "page_view_count": 3,
    "unique_route_count": 2,
    "click_count": 12,
    "field_change_count": 5,
    "flow_success_count": 0,
    "flow_failure_count": 3,
    "error_event_count": 4,
    "retry_count": 2,
    "rage_click_count": 6,          # High frustration indicator
    "session_duration_ms": 150000,
    "total_dwell_ms": 80000,
    "avg_inter_event_gap_ms": 800
}

print(f"Sending payload to endpoint: frustration-ensemble-endpoint...")

response = runtime_client.invoke_endpoint(
    EndpointName="frustration-ensemble-endpoint",
    ContentType="application/json",
    Body=json.dumps(test_payload)
)

result = json.loads(response['Body'].read().decode())

print("\n--- API Response ---")
print(json.dumps(result, indent=2))

In [ ]:
sess = sagemaker.Session()

endpoint_name = "frustration-ensemble-endpoint"

print(f"Deleting endpoint: {endpoint_name}...")
sess.delete_endpoint(endpoint_name=endpoint_name)

print("Endpoint successfully deleted!")

In [4]:
%%writefile inference.py

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import argparse
import boto3
import json
import numpy as np
import pandas as pd

import preprocess as prep

# --- Constants & S3 Config ---
BUCKET_NAME = "sagemaker-studio-i0gutcxdy"
RESULTS_PREFIX = "frustration-model/results"
TRAIN_TEST_PREFIX = "frustration-model"
ENDPOINT_NAME = "frustration-ensemble-endpoint"

# Provided Thresholds (Anchored to a 9.0 Score)
AE_THRESH = 0.546781
IF_THRESH = 0.023487
GMM_THRESH = -8.274406

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def process_raw_data(raw_input_path, do_split_upload=False):
    print(f"\n--- 1. Processing Raw Telemetry Data ---")
    print(f"Loading raw telemetry from: {raw_input_path}")
    
    loader = prep.S3DataLoader(source=raw_input_path)
    raw_events = loader.fetchRawLogs()
    if not raw_events:
        raise RuntimeError("No telemetry events loaded.")

    aggregator = prep.SessionAggregator()
    aggregator.ingest_many(raw_events)
    sessions = aggregator.groupBySession()
    
    # Extract features but DO NOT normalize locally
    extractor = prep.FeatureExtractor(do_normalize=False)
    X_raw, feature_names, session_ids, session_timestamps = prep.build_feature_matrix(sessions, extractor)
    
    df_full = pd.DataFrame(X_raw, columns=feature_names)
    df_full['sessionId'] = session_ids
    df_full['timestamp'] = session_timestamps
    
    print(f"Successfully processed {len(df_full)} sessions.")
    
    if do_split_upload:
        print("\n--- Optional: Splitting Data and Uploading to S3 ---")
        train_data, test_data = np.split(df_full.sample(frac=1, random_state=42), [int(0.7 * len(df_full))])
        
        out_dir = './out'
        os.makedirs(out_dir, exist_ok=True)
        train_local = os.path.join(out_dir, "train.csv")
        test_local = os.path.join(out_dir, "test.csv")
        
        train_data.to_csv(train_local, index=False)
        test_data.to_csv(test_local, index=False)
        
        s3_client = boto3.client('s3')
        s3_client.upload_file(train_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/traindir/train.csv")
        s3_client.upload_file(test_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/testdir/test.csv")
        print(f"[OK] 70/30 Train/Test split uploaded to S3")
        
    return df_full, feature_names

def run_inference(args):
    df_full, feature_names = process_raw_data(args.input_raw, args.split_upload)
    
    session_ids = df_full['sessionId'].values
    timestamps = df_full['timestamp'].values
    features_df = df_full[feature_names] 
    
    print(f"\n--- 2. Fetching Raw Scores from SageMaker Endpoint ---")
    runtime_client = boto3.client('sagemaker-runtime')
    
    results = []
    total_sessions = len(features_df)

    for index, row in features_df.iterrows():
        response = runtime_client.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='application/json',
            Body=json.dumps(row.to_dict())
        )
        results.append(json.loads(response['Body'].read().decode()))
        
        if (index + 1) % 50 == 0:
            print(f"Processed {index + 1}/{total_sessions} sessions...")

    # Extract the raw batch scores from the endpoint's response
    ae_mse = np.array([r['breakdown']['ae_raw_mse'] for r in results])
    if_scores = np.array([r['breakdown']['if_raw_score'] for r in results])
    gmm_scores = np.array([r['breakdown']['gmm_raw_score'] for r in results])

    print("\n--- 3. Calculating 0-10 Severity Scores ---")
    
    # AE Scaling
    ae_min = np.min(ae_mse)
    ae_scale_factor = (AE_THRESH - ae_min) if (AE_THRESH - ae_min) > 0 else 1e-6
    ae_scores_0_to_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    # IF Scaling
    if_max = np.max(if_scores)
    if_scale_factor = (if_max - IF_THRESH) if (if_max - IF_THRESH) > 0 else 1e-6
    if_scores_0_to_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    # GMM Scaling
    gmm_min = np.min(gmm_scores)
    gmm_scale_factor = (GMM_THRESH - gmm_min) if (GMM_THRESH - gmm_min) > 0 else 1e-6
    gmm_scores_0_to_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    # Ensemble Scaling
    ensemble_scores_0_to_10 = (ae_scores_0_to_10 + if_scores_0_to_10 + gmm_scores_0_to_10) / 3.0

    print("Formatting output CSVs...")
    def build_df(scores):
        return pd.DataFrame({
            'sessionId': session_ids,
            'frustrationScore': np.round(scores, 2),
            'severity': [calculate_severity(s) for s in scores],
            'timestamp': timestamps
        })

    df_ae = build_df(ae_scores_0_to_10)
    df_if = build_df(if_scores_0_to_10)
    df_gmm = build_df(gmm_scores_0_to_10)
    df_ensemble = build_df(ensemble_scores_0_to_10)

    # 4. Save Locally
    out_dir = './inference_results'
    os.makedirs(out_dir, exist_ok=True)
    
    ae_out = os.path.join(out_dir, 'ae_predictions.csv')
    if_out = os.path.join(out_dir, 'if_predictions.csv')
    gmm_out = os.path.join(out_dir, 'gmm_predictions.csv')
    ens_out = os.path.join(out_dir, 'ensemble_predictions.csv')
    
    df_ae.to_csv(ae_out, index=False)
    df_if.to_csv(if_out, index=False)
    df_gmm.to_csv(gmm_out, index=False)
    df_ensemble.to_csv(ens_out, index=False)
    
    print(f"\nPredictions saved locally to {out_dir}/")
    print(f"Ensemble High Severity Count : {sum(df_ensemble['severity'] == 'High')}")

    # 5. Upload Results to S3
    print(f"\n--- 4. Uploading Results to S3 ---")
    s3_client = boto3.client('s3')
    files_to_upload = {
        ae_out: f"{RESULTS_PREFIX}/ae_predictions.csv",
        if_out: f"{RESULTS_PREFIX}/if_predictions.csv",
        gmm_out: f"{RESULTS_PREFIX}/gmm_predictions.csv",
        ens_out: f"{RESULTS_PREFIX}/ensemble_predictions.csv"
    }
    
    for local_file, s3_key in files_to_upload.items():
        s3_client.upload_file(local_file, BUCKET_NAME, s3_key)
        print(f"Uploaded -> s3://{BUCKET_NAME}/{s3_key}")
        
    print("\n[SUCCESS] End-to-end API Inference Complete.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-raw', type=str, required=True, help="Path to raw NDJSON/JSONL")
    parser.add_argument('--split-upload', action='store_true', help="Split 70/30 and upload train/test CSVs to S3")
    
    args = parser.parse_args()
    run_inference(args)

Writing inference.py


In [4]:
%%writefile inference.py

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import argparse
import boto3
import json
import numpy as np
import pandas as pd

import preprocess as prep

# --- Constants & S3 Config ---
BUCKET_NAME = "sagemaker-studio-i0gutcxdy"
RESULTS_PREFIX = "frustration-model/results"
TRAIN_TEST_PREFIX = "frustration-model"
ENDPOINT_NAME = "frustration-ensemble-endpoint"

# Provided Thresholds (Anchored to a 9.0 Score)
AE_THRESH = 0.546781
IF_THRESH = 0.023487
GMM_THRESH = -8.274406

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def process_raw_data(raw_input_path, do_split_upload=False):
    print(f"\n--- 1. Processing Raw Telemetry Data ---")
    print(f"Loading raw telemetry from: {raw_input_path}")
    
    loader = prep.S3DataLoader(source=raw_input_path)
    raw_events = loader.fetchRawLogs()
    if not raw_events:
        raise RuntimeError("No telemetry events loaded.")

    aggregator = prep.SessionAggregator()
    aggregator.ingest_many(raw_events)
    sessions = aggregator.groupBySession()
    
    # Extract features but DO NOT normalize locally
    extractor = prep.FeatureExtractor(do_normalize=False)
    X_raw, feature_names, session_ids, session_timestamps = prep.build_feature_matrix(sessions, extractor)
    
    df_full = pd.DataFrame(X_raw, columns=feature_names)
    df_full['sessionId'] = session_ids
    df_full['timestamp'] = session_timestamps
    
    print(f"Successfully processed {len(df_full)} sessions.")
    
    if do_split_upload:
        print("\n--- Optional: Splitting Data and Uploading to S3 ---")
        train_data, test_data = np.split(df_full.sample(frac=1, random_state=42), [int(0.7 * len(df_full))])
        
        out_dir = './out'
        os.makedirs(out_dir, exist_ok=True)
        train_local = os.path.join(out_dir, "train.csv")
        test_local = os.path.join(out_dir, "test.csv")
        
        train_data.to_csv(train_local, index=False)
        test_data.to_csv(test_local, index=False)
        
        s3_client = boto3.client('s3')
        s3_client.upload_file(train_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/traindir/train.csv")
        s3_client.upload_file(test_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/testdir/test.csv")
        print(f"[OK] 70/30 Train/Test split uploaded to S3")
        
    return df_full, feature_names

def run_inference(args):
    df_full, feature_names = process_raw_data(args.input_raw, args.split_upload)
    
    session_ids = df_full['sessionId'].values
    timestamps = df_full['timestamp'].values
    features_df = df_full[feature_names] 
    
    print(f"\n--- 2. Fetching Raw Scores from SageMaker Endpoint ---")
    runtime_client = boto3.client('sagemaker-runtime')
    
    results = []
    total_sessions = len(features_df)

    for index, row in features_df.iterrows():
        response = runtime_client.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='application/json',
            Body=json.dumps(row.to_dict())
        )
        results.append(json.loads(response['Body'].read().decode()))
        
        if (index + 1) % 50 == 0:
            print(f"Processed {index + 1}/{total_sessions} sessions...")

    # Extract the raw batch scores from the endpoint's response
    ae_mse = np.array([r['breakdown']['ae_raw_mse'] for r in results])
    if_scores = np.array([r['breakdown']['if_raw_score'] for r in results])
    gmm_scores = np.array([r['breakdown']['gmm_raw_score'] for r in results])

    print("\n--- 3. Calculating 0-10 Severity Scores ---")
    
    # AE Scaling
    ae_min = np.min(ae_mse)
    ae_scale_factor = (AE_THRESH - ae_min) if (AE_THRESH - ae_min) > 0 else 1e-6
    ae_scores_0_to_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    # IF Scaling
    if_max = np.max(if_scores)
    if_scale_factor = (if_max - IF_THRESH) if (if_max - IF_THRESH) > 0 else 1e-6
    if_scores_0_to_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    # GMM Scaling
    gmm_min = np.min(gmm_scores)
    gmm_scale_factor = (GMM_THRESH - gmm_min) if (GMM_THRESH - gmm_min) > 0 else 1e-6
    gmm_scores_0_to_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    # Ensemble Scaling
    ensemble_scores_0_to_10 = (ae_scores_0_to_10 + if_scores_0_to_10 + gmm_scores_0_to_10) / 3.0

    print("Formatting output CSVs...")
    def build_df(scores):
        return pd.DataFrame({
            'sessionId': session_ids,
            'frustrationScore': np.round(scores, 2),
            'severity': [calculate_severity(s) for s in scores],
            'timestamp': timestamps
        })

    df_ae = build_df(ae_scores_0_to_10)
    df_if = build_df(if_scores_0_to_10)
    df_gmm = build_df(gmm_scores_0_to_10)
    df_ensemble = build_df(ensemble_scores_0_to_10)

    # 4. Save Locally
    out_dir = './inference_results'
    os.makedirs(out_dir, exist_ok=True)
    
    ae_out = os.path.join(out_dir, 'ae_predictions.csv')
    if_out = os.path.join(out_dir, 'if_predictions.csv')
    gmm_out = os.path.join(out_dir, 'gmm_predictions.csv')
    ens_out = os.path.join(out_dir, 'ensemble_predictions.csv')
    
    df_ae.to_csv(ae_out, index=False)
    df_if.to_csv(if_out, index=False)
    df_gmm.to_csv(gmm_out, index=False)
    df_ensemble.to_csv(ens_out, index=False)
    
    print(f"\nPredictions saved locally to {out_dir}/")
    print(f"Ensemble High Severity Count : {sum(df_ensemble['severity'] == 'High')}")

    # 5. Upload Results to S3
    print(f"\n--- 4. Uploading Results to S3 ---")
    s3_client = boto3.client('s3')
    files_to_upload = {
        ae_out: f"{RESULTS_PREFIX}/ae_predictions.csv",
        if_out: f"{RESULTS_PREFIX}/if_predictions.csv",
        gmm_out: f"{RESULTS_PREFIX}/gmm_predictions.csv",
        ens_out: f"{RESULTS_PREFIX}/ensemble_predictions.csv"
    }
    
    for local_file, s3_key in files_to_upload.items():
        s3_client.upload_file(local_file, BUCKET_NAME, s3_key)
        print(f"Uploaded -> s3://{BUCKET_NAME}/{s3_key}")
        
    print("\n[SUCCESS] End-to-end API Inference Complete.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-raw', type=str, required=True, help="Path to raw NDJSON/JSONL")
    parser.add_argument('--split-upload', action='store_true', help="Split 70/30 and upload train/test CSVs to S3")
    
    args = parser.parse_args()
    run_inference(args)

Writing inference.py


In [4]:
%%writefile inference.py

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import argparse
import boto3
import json
import numpy as np
import pandas as pd

import preprocess as prep

# --- Constants & S3 Config ---
BUCKET_NAME = "sagemaker-studio-i0gutcxdy"
RESULTS_PREFIX = "frustration-model/results"
TRAIN_TEST_PREFIX = "frustration-model"
ENDPOINT_NAME = "frustration-ensemble-endpoint"

# Provided Thresholds (Anchored to a 9.0 Score)
AE_THRESH = 0.546781
IF_THRESH = 0.023487
GMM_THRESH = -8.274406

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def process_raw_data(raw_input_path, do_split_upload=False):
    print(f"\n--- 1. Processing Raw Telemetry Data ---")
    print(f"Loading raw telemetry from: {raw_input_path}")
    
    loader = prep.S3DataLoader(source=raw_input_path)
    raw_events = loader.fetchRawLogs()
    if not raw_events:
        raise RuntimeError("No telemetry events loaded.")

    aggregator = prep.SessionAggregator()
    aggregator.ingest_many(raw_events)
    sessions = aggregator.groupBySession()
    
    # Extract features but DO NOT normalize locally
    extractor = prep.FeatureExtractor(do_normalize=False)
    X_raw, feature_names, session_ids, session_timestamps = prep.build_feature_matrix(sessions, extractor)
    
    df_full = pd.DataFrame(X_raw, columns=feature_names)
    df_full['sessionId'] = session_ids
    df_full['timestamp'] = session_timestamps
    
    print(f"Successfully processed {len(df_full)} sessions.")
    
    if do_split_upload:
        print("\n--- Optional: Splitting Data and Uploading to S3 ---")
        train_data, test_data = np.split(df_full.sample(frac=1, random_state=42), [int(0.7 * len(df_full))])
        
        out_dir = './out'
        os.makedirs(out_dir, exist_ok=True)
        train_local = os.path.join(out_dir, "train.csv")
        test_local = os.path.join(out_dir, "test.csv")
        
        train_data.to_csv(train_local, index=False)
        test_data.to_csv(test_local, index=False)
        
        s3_client = boto3.client('s3')
        s3_client.upload_file(train_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/traindir/train.csv")
        s3_client.upload_file(test_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/testdir/test.csv")
        print(f"[OK] 70/30 Train/Test split uploaded to S3")
        
    return df_full, feature_names

def run_inference(args):
    df_full, feature_names = process_raw_data(args.input_raw, args.split_upload)
    
    session_ids = df_full['sessionId'].values
    timestamps = df_full['timestamp'].values
    features_df = df_full[feature_names] 
    
    print(f"\n--- 2. Fetching Raw Scores from SageMaker Endpoint ---")
    runtime_client = boto3.client('sagemaker-runtime')
    
    results = []
    total_sessions = len(features_df)

    for index, row in features_df.iterrows():
        response = runtime_client.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='application/json',
            Body=json.dumps(row.to_dict())
        )
        results.append(json.loads(response['Body'].read().decode()))
        
        if (index + 1) % 50 == 0:
            print(f"Processed {index + 1}/{total_sessions} sessions...")

    # Extract the raw batch scores from the endpoint's response
    ae_mse = np.array([r['breakdown']['ae_raw_mse'] for r in results])
    if_scores = np.array([r['breakdown']['if_raw_score'] for r in results])
    gmm_scores = np.array([r['breakdown']['gmm_raw_score'] for r in results])

    print("\n--- 3. Calculating 0-10 Severity Scores ---")
    
    # AE Scaling
    ae_min = np.min(ae_mse)
    ae_scale_factor = (AE_THRESH - ae_min) if (AE_THRESH - ae_min) > 0 else 1e-6
    ae_scores_0_to_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    # IF Scaling
    if_max = np.max(if_scores)
    if_scale_factor = (if_max - IF_THRESH) if (if_max - IF_THRESH) > 0 else 1e-6
    if_scores_0_to_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    # GMM Scaling
    gmm_min = np.min(gmm_scores)
    gmm_scale_factor = (GMM_THRESH - gmm_min) if (GMM_THRESH - gmm_min) > 0 else 1e-6
    gmm_scores_0_to_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    # Ensemble Scaling
    ensemble_scores_0_to_10 = (ae_scores_0_to_10 + if_scores_0_to_10 + gmm_scores_0_to_10) / 3.0

    print("Formatting output CSVs...")
    def build_df(scores):
        return pd.DataFrame({
            'sessionId': session_ids,
            'frustrationScore': np.round(scores, 2),
            'severity': [calculate_severity(s) for s in scores],
            'timestamp': timestamps
        })

    df_ae = build_df(ae_scores_0_to_10)
    df_if = build_df(if_scores_0_to_10)
    df_gmm = build_df(gmm_scores_0_to_10)
    df_ensemble = build_df(ensemble_scores_0_to_10)

    # 4. Save Locally
    out_dir = './inference_results'
    os.makedirs(out_dir, exist_ok=True)
    
    ae_out = os.path.join(out_dir, 'ae_predictions.csv')
    if_out = os.path.join(out_dir, 'if_predictions.csv')
    gmm_out = os.path.join(out_dir, 'gmm_predictions.csv')
    ens_out = os.path.join(out_dir, 'ensemble_predictions.csv')
    
    df_ae.to_csv(ae_out, index=False)
    df_if.to_csv(if_out, index=False)
    df_gmm.to_csv(gmm_out, index=False)
    df_ensemble.to_csv(ens_out, index=False)
    
    print(f"\nPredictions saved locally to {out_dir}/")
    print(f"Ensemble High Severity Count : {sum(df_ensemble['severity'] == 'High')}")

    # 5. Upload Results to S3
    print(f"\n--- 4. Uploading Results to S3 ---")
    s3_client = boto3.client('s3')
    files_to_upload = {
        ae_out: f"{RESULTS_PREFIX}/ae_predictions.csv",
        if_out: f"{RESULTS_PREFIX}/if_predictions.csv",
        gmm_out: f"{RESULTS_PREFIX}/gmm_predictions.csv",
        ens_out: f"{RESULTS_PREFIX}/ensemble_predictions.csv"
    }
    
    for local_file, s3_key in files_to_upload.items():
        s3_client.upload_file(local_file, BUCKET_NAME, s3_key)
        print(f"Uploaded -> s3://{BUCKET_NAME}/{s3_key}")
        
    print("\n[SUCCESS] End-to-end API Inference Complete.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-raw', type=str, required=True, help="Path to raw NDJSON/JSONL")
    parser.add_argument('--split-upload', action='store_true', help="Split 70/30 and upload train/test CSVs to S3")
    
    args = parser.parse_args()
    run_inference(args)

Writing inference.py


In [1]:
%%writefile requirements.txt
tensorflow>=2.13.0
scikit-learn==1.2.2
pyod==1.1.2
pandas>=2.0.0

Writing requirements.txt


In [2]:
%%writefile serve.py
import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from pyod.models.gmm import GMM

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def model_fn(model_dir):
    print("Loading Keras AutoEncoder...")
    ae_path = os.path.join(model_dir, 'autoencoder_model.keras')
    autoencoder = tf.keras.models.load_model(ae_path)
    
    print("Loading Isolation Forest...")
    iso_path = os.path.join(model_dir, 'isolation_forest.pkl')
    isolation_forest = joblib.load(iso_path)
    
    print("Loading GMM...")
    gmm_path = os.path.join(model_dir, 'gmm_model.pkl')
    gmm_model = joblib.load(gmm_path)
    
    print("Loading Scaler...")
    scaler_path = os.path.join(model_dir, 'scaler.pkl')
    scaler = joblib.load(scaler_path)
    
    print("Loading Metadata and Thresholds...")
    metadata_path = os.path.join(model_dir, 'model_metadata.joblib')
    metadata = joblib.load(metadata_path)
    
    return {
        'autoencoder': autoencoder,
        'isolation_forest': isolation_forest,
        'gmm_model': gmm_model,
        'scaler': scaler,
        'metadata': metadata
    }

def input_fn(request_body, request_content_type):
    if request_content_type == 'application/json':
        data = json.loads(request_body)
        return pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}. Expected 'application/json'.")

def predict_fn(input_data, model_dict):
    autoencoder = model_dict['autoencoder']
    isolation_forest = model_dict['isolation_forest']
    gmm_model = model_dict['gmm_model']
    scaler = model_dict['scaler']
    metadata = model_dict['metadata']
    
    ae_thresh = metadata['ae_threshold']
    iso_thresh = metadata['iso_threshold']
    gmm_thresh = metadata['gmm_threshold']
    
    features = input_data[metadata['feature_names']]
    X_scaled = scaler.transform(features).astype(np.float32)
    
    reconstructed = autoencoder.predict(X_scaled, verbose=0)
    ae_mse = np.mean(np.square(X_scaled - reconstructed), axis=1)
    if_scores = isolation_forest.decision_function(X_scaled)
    gmm_scores = gmm_model.decision_function(X_scaled)
    
    ae_min = metadata.get('ae_min_error', 0.0) 
    ae_scale_factor = (ae_thresh - ae_min) if (ae_thresh - ae_min) > 0 else 1e-6
    ae_score_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    if_max = 0.15 
    if_scale_factor = (if_max - iso_thresh) if (if_max - iso_thresh) > 0 else 1e-6
    if_score_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    gmm_min = np.min(gmm_scores) 
    gmm_scale_factor = (gmm_thresh - gmm_min) if (gmm_thresh - gmm_min) > 0 else 1e-6
    gmm_score_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    ensemble_score_10 = float((ae_score_10[0] + if_score_10[0] + gmm_score_10[0]) / 3.0)
    
    return {
        "frustrationScore": round(ensemble_score_10, 2),
        "severity": calculate_severity(ensemble_score_10),
        "breakdown": {
            "ae_raw_mse": float(ae_mse[0]),
            "if_raw_score": float(if_scores[0]),
            "gmm_raw_score": float(gmm_scores[0])
        }
    }

def output_fn(prediction, accept):
    if accept == 'application/json':
        return json.dumps(prediction), accept
    else:
        raise ValueError(f"Accept header must be 'application/json', got: {accept}")

Overwriting serve.py


In [3]:
sess = sagemaker.Session()
role = sagemaker.get_execution_role() 

current_dir = os.path.abspath(os.getcwd())
req_path = os.path.join(current_dir, "requirements.txt")
serve_path = os.path.join(current_dir, "serve.py") 

model_s3_path = "s3://sagemaker-studio-i0gutcxdy/frustration-model/models/model.tar.gz"

print(f"Using requirements from: {req_path}")
print(f"Using entry point from: {serve_path}")
print("Configuring the Ensemble Model for SageMaker...")

ensemble_model = SKLearnModel(
    model_data=model_s3_path,
    role=role,
    entry_point=serve_path,
    dependencies=[req_path],
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sess
)
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5
)

print("Deploying Serverless endpoint)")
predictor = ensemble_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="frustration-serverless-endpoint"
)

print(f"\n[SUCCESS] Serverless Endpoint deployed at: {predictor.endpoint_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Using requirements from: /home/sagemaker-user/deployment/requirements.txt
Using entry point from: /home/sagemaker-user/deployment/serve.py
Configuring the Ensemble Model for SageMaker...
Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)
------------------------------------------

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:29                                                                                   │
│                                                                                                  │
│   26 )                                                                                           │
│   27                                                                                             │
│   28 print("Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)")                     │
│ ❱ 29 predictor = ensemble_model.deploy(                                                          │
│   30 │   initial_instance_count=1,                                                               │
│   31 │   instance_type="ml.m5.large",                                                            │
│   32 │   endpoint_name="frustration-ensemble-endpoint"                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model.py:1814 in deploy                        │
│                                                                                                  │
│   1811 │   │   │   │   )                                                                         │
│   1812 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1813 │   │   │   else:                                                                         │
│ ❱ 1814 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1815 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1816 │   │   │   │   │   production_variants=[production_variant],                             │
│   1817 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:6033 in                             │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6030 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6031 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6032 │   │                                                                                     │
│ ❱ 6033 │   │   return self.create_endpoint(                                                      │
│   6034 │   │   │   endpoint_name=name,                                                           │
│   6035 │   │   │   config_name=name,                                                             │
│   6036 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:4867 in create_endpoint             │
│                                                                                                  │
│   4864 │   │   │   │   self.endpoint_arn = res["EndpointArn"]                                    │
│   4865 │   │   │                                                                                 │
│   4866 │   │   │   if wait:                                                                      │
│ ❱ 4867 │   │   │   │   self.wait_for_endpoint(endpoint_name, live_logging=live_logging)          │
│   4868 │   │   │   return endpoint_name                                                          │
│   4869 │   │   except Exception as e:                      

In [ ]:
import boto3
import json

runtime_client = boto3.client('sagemaker-runtime')

test_payload = {
    "event_count": 45,
    "page_view_count": 3,
    "unique_route_count": 2,
    "click_count": 12,
    "field_change_count": 5,
    "flow_success_count": 0,
    "flow_failure_count": 3,
    "error_event_count": 4,
    "retry_count": 2,
    "rage_click_count": 6,          # High frustration indicator
    "session_duration_ms": 150000,
    "total_dwell_ms": 80000,
    "avg_inter_event_gap_ms": 800
}

print(f"Sending payload to endpoint: frustration-ensemble-endpoint...")

response = runtime_client.invoke_endpoint(
    EndpointName="frustration-ensemble-endpoint",
    ContentType="application/json",
    Body=json.dumps(test_payload)
)

result = json.loads(response['Body'].read().decode())

print("\n--- API Response ---")
print(json.dumps(result, indent=2))

In [ ]:
sess = sagemaker.Session()

endpoint_name = "frustration-ensemble-endpoint"

print(f"Deleting endpoint: {endpoint_name}...")
sess.delete_endpoint(endpoint_name=endpoint_name)

print("Endpoint successfully deleted!")

In [4]:
%%writefile inference.py

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import argparse
import boto3
import json
import numpy as np
import pandas as pd

import preprocess as prep

# --- Constants & S3 Config ---
BUCKET_NAME = "sagemaker-studio-i0gutcxdy"
RESULTS_PREFIX = "frustration-model/results"
TRAIN_TEST_PREFIX = "frustration-model"
ENDPOINT_NAME = "frustration-ensemble-endpoint"

# Provided Thresholds (Anchored to a 9.0 Score)
AE_THRESH = 0.546781
IF_THRESH = 0.023487
GMM_THRESH = -8.274406

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def process_raw_data(raw_input_path, do_split_upload=False):
    print(f"\n--- 1. Processing Raw Telemetry Data ---")
    print(f"Loading raw telemetry from: {raw_input_path}")
    
    loader = prep.S3DataLoader(source=raw_input_path)
    raw_events = loader.fetchRawLogs()
    if not raw_events:
        raise RuntimeError("No telemetry events loaded.")

    aggregator = prep.SessionAggregator()
    aggregator.ingest_many(raw_events)
    sessions = aggregator.groupBySession()
    
    # Extract features but DO NOT normalize locally
    extractor = prep.FeatureExtractor(do_normalize=False)
    X_raw, feature_names, session_ids, session_timestamps = prep.build_feature_matrix(sessions, extractor)
    
    df_full = pd.DataFrame(X_raw, columns=feature_names)
    df_full['sessionId'] = session_ids
    df_full['timestamp'] = session_timestamps
    
    print(f"Successfully processed {len(df_full)} sessions.")
    
    if do_split_upload:
        print("\n--- Optional: Splitting Data and Uploading to S3 ---")
        train_data, test_data = np.split(df_full.sample(frac=1, random_state=42), [int(0.7 * len(df_full))])
        
        out_dir = './out'
        os.makedirs(out_dir, exist_ok=True)
        train_local = os.path.join(out_dir, "train.csv")
        test_local = os.path.join(out_dir, "test.csv")
        
        train_data.to_csv(train_local, index=False)
        test_data.to_csv(test_local, index=False)
        
        s3_client = boto3.client('s3')
        s3_client.upload_file(train_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/traindir/train.csv")
        s3_client.upload_file(test_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/testdir/test.csv")
        print(f"[OK] 70/30 Train/Test split uploaded to S3")
        
    return df_full, feature_names

def run_inference(args):
    df_full, feature_names = process_raw_data(args.input_raw, args.split_upload)
    
    session_ids = df_full['sessionId'].values
    timestamps = df_full['timestamp'].values
    features_df = df_full[feature_names] 
    
    print(f"\n--- 2. Fetching Raw Scores from SageMaker Endpoint ---")
    runtime_client = boto3.client('sagemaker-runtime')
    
    results = []
    total_sessions = len(features_df)

    for index, row in features_df.iterrows():
        response = runtime_client.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='application/json',
            Body=json.dumps(row.to_dict())
        )
        results.append(json.loads(response['Body'].read().decode()))
        
        if (index + 1) % 50 == 0:
            print(f"Processed {index + 1}/{total_sessions} sessions...")

    # Extract the raw batch scores from the endpoint's response
    ae_mse = np.array([r['breakdown']['ae_raw_mse'] for r in results])
    if_scores = np.array([r['breakdown']['if_raw_score'] for r in results])
    gmm_scores = np.array([r['breakdown']['gmm_raw_score'] for r in results])

    print("\n--- 3. Calculating 0-10 Severity Scores ---")
    
    # AE Scaling
    ae_min = np.min(ae_mse)
    ae_scale_factor = (AE_THRESH - ae_min) if (AE_THRESH - ae_min) > 0 else 1e-6
    ae_scores_0_to_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    # IF Scaling
    if_max = np.max(if_scores)
    if_scale_factor = (if_max - IF_THRESH) if (if_max - IF_THRESH) > 0 else 1e-6
    if_scores_0_to_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    # GMM Scaling
    gmm_min = np.min(gmm_scores)
    gmm_scale_factor = (GMM_THRESH - gmm_min) if (GMM_THRESH - gmm_min) > 0 else 1e-6
    gmm_scores_0_to_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    # Ensemble Scaling
    ensemble_scores_0_to_10 = (ae_scores_0_to_10 + if_scores_0_to_10 + gmm_scores_0_to_10) / 3.0

    print("Formatting output CSVs...")
    def build_df(scores):
        return pd.DataFrame({
            'sessionId': session_ids,
            'frustrationScore': np.round(scores, 2),
            'severity': [calculate_severity(s) for s in scores],
            'timestamp': timestamps
        })

    df_ae = build_df(ae_scores_0_to_10)
    df_if = build_df(if_scores_0_to_10)
    df_gmm = build_df(gmm_scores_0_to_10)
    df_ensemble = build_df(ensemble_scores_0_to_10)

    # 4. Save Locally
    out_dir = './inference_results'
    os.makedirs(out_dir, exist_ok=True)
    
    ae_out = os.path.join(out_dir, 'ae_predictions.csv')
    if_out = os.path.join(out_dir, 'if_predictions.csv')
    gmm_out = os.path.join(out_dir, 'gmm_predictions.csv')
    ens_out = os.path.join(out_dir, 'ensemble_predictions.csv')
    
    df_ae.to_csv(ae_out, index=False)
    df_if.to_csv(if_out, index=False)
    df_gmm.to_csv(gmm_out, index=False)
    df_ensemble.to_csv(ens_out, index=False)
    
    print(f"\nPredictions saved locally to {out_dir}/")
    print(f"Ensemble High Severity Count : {sum(df_ensemble['severity'] == 'High')}")

    # 5. Upload Results to S3
    print(f"\n--- 4. Uploading Results to S3 ---")
    s3_client = boto3.client('s3')
    files_to_upload = {
        ae_out: f"{RESULTS_PREFIX}/ae_predictions.csv",
        if_out: f"{RESULTS_PREFIX}/if_predictions.csv",
        gmm_out: f"{RESULTS_PREFIX}/gmm_predictions.csv",
        ens_out: f"{RESULTS_PREFIX}/ensemble_predictions.csv"
    }
    
    for local_file, s3_key in files_to_upload.items():
        s3_client.upload_file(local_file, BUCKET_NAME, s3_key)
        print(f"Uploaded -> s3://{BUCKET_NAME}/{s3_key}")
        
    print("\n[SUCCESS] End-to-end API Inference Complete.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-raw', type=str, required=True, help="Path to raw NDJSON/JSONL")
    parser.add_argument('--split-upload', action='store_true', help="Split 70/30 and upload train/test CSVs to S3")
    
    args = parser.parse_args()
    run_inference(args)

Writing inference.py


In [1]:
%%writefile requirements.txt
tensorflow>=2.13.0
scikit-learn==1.2.2
pyod==1.1.2
pandas>=2.0.0

Writing requirements.txt


In [2]:
%%writefile serve.py
import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from pyod.models.gmm import GMM

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def model_fn(model_dir):
    print("Loading Keras AutoEncoder...")
    ae_path = os.path.join(model_dir, 'autoencoder_model.keras')
    autoencoder = tf.keras.models.load_model(ae_path)
    
    print("Loading Isolation Forest...")
    iso_path = os.path.join(model_dir, 'isolation_forest.pkl')
    isolation_forest = joblib.load(iso_path)
    
    print("Loading GMM...")
    gmm_path = os.path.join(model_dir, 'gmm_model.pkl')
    gmm_model = joblib.load(gmm_path)
    
    print("Loading Scaler...")
    scaler_path = os.path.join(model_dir, 'scaler.pkl')
    scaler = joblib.load(scaler_path)
    
    print("Loading Metadata and Thresholds...")
    metadata_path = os.path.join(model_dir, 'model_metadata.joblib')
    metadata = joblib.load(metadata_path)
    
    return {
        'autoencoder': autoencoder,
        'isolation_forest': isolation_forest,
        'gmm_model': gmm_model,
        'scaler': scaler,
        'metadata': metadata
    }

def input_fn(request_body, request_content_type):
    if request_content_type == 'application/json':
        data = json.loads(request_body)
        return pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}. Expected 'application/json'.")

def predict_fn(input_data, model_dict):
    autoencoder = model_dict['autoencoder']
    isolation_forest = model_dict['isolation_forest']
    gmm_model = model_dict['gmm_model']
    scaler = model_dict['scaler']
    metadata = model_dict['metadata']
    
    ae_thresh = metadata['ae_threshold']
    iso_thresh = metadata['iso_threshold']
    gmm_thresh = metadata['gmm_threshold']
    
    features = input_data[metadata['feature_names']]
    X_scaled = scaler.transform(features).astype(np.float32)
    
    reconstructed = autoencoder.predict(X_scaled, verbose=0)
    ae_mse = np.mean(np.square(X_scaled - reconstructed), axis=1)
    if_scores = isolation_forest.decision_function(X_scaled)
    gmm_scores = gmm_model.decision_function(X_scaled)
    
    ae_min = metadata.get('ae_min_error', 0.0) 
    ae_scale_factor = (ae_thresh - ae_min) if (ae_thresh - ae_min) > 0 else 1e-6
    ae_score_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    if_max = 0.15 
    if_scale_factor = (if_max - iso_thresh) if (if_max - iso_thresh) > 0 else 1e-6
    if_score_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    gmm_min = np.min(gmm_scores) 
    gmm_scale_factor = (gmm_thresh - gmm_min) if (gmm_thresh - gmm_min) > 0 else 1e-6
    gmm_score_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    ensemble_score_10 = float((ae_score_10[0] + if_score_10[0] + gmm_score_10[0]) / 3.0)
    
    return {
        "frustrationScore": round(ensemble_score_10, 2),
        "severity": calculate_severity(ensemble_score_10),
        "breakdown": {
            "ae_raw_mse": float(ae_mse[0]),
            "if_raw_score": float(if_scores[0]),
            "gmm_raw_score": float(gmm_scores[0])
        }
    }

def output_fn(prediction, accept):
    if accept == 'application/json':
        return json.dumps(prediction), accept
    else:
        raise ValueError(f"Accept header must be 'application/json', got: {accept}")

Overwriting serve.py


In [3]:
sess = sagemaker.Session()
role = sagemaker.get_execution_role() 

current_dir = os.path.abspath(os.getcwd())
req_path = os.path.join(current_dir, "requirements.txt")
serve_path = os.path.join(current_dir, "serve.py") 

model_s3_path = "s3://sagemaker-studio-i0gutcxdy/frustration-model/models/model.tar.gz"

print(f"Using requirements from: {req_path}")
print(f"Using entry point from: {serve_path}")
print("Configuring the Ensemble Model for SageMaker...")

ensemble_model = SKLearnModel(
    model_data=model_s3_path,
    role=role,
    entry_point=serve_path,
    dependencies=[req_path],
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sess
)
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5
)

print("Deploying Serverless endpoint)")
predictor = ensemble_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="frustration-serverless-endpoint"
)

print(f"\n[SUCCESS] Serverless Endpoint deployed at: {predictor.endpoint_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Using requirements from: /home/sagemaker-user/deployment/requirements.txt
Using entry point from: /home/sagemaker-user/deployment/serve.py
Configuring the Ensemble Model for SageMaker...
Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)
------------------------------------------

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:29                                                                                   │
│                                                                                                  │
│   26 )                                                                                           │
│   27                                                                                             │
│   28 print("Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)")                     │
│ ❱ 29 predictor = ensemble_model.deploy(                                                          │
│   30 │   initial_instance_count=1,                                                               │
│   31 │   instance_type="ml.m5.large",                                                            │
│   32 │   endpoint_name="frustration-ensemble-endpoint"                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model.py:1814 in deploy                        │
│                                                                                                  │
│   1811 │   │   │   │   )                                                                         │
│   1812 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1813 │   │   │   else:                                                                         │
│ ❱ 1814 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1815 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1816 │   │   │   │   │   production_variants=[production_variant],                             │
│   1817 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:6033 in                             │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6030 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6031 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6032 │   │                                                                                     │
│ ❱ 6033 │   │   return self.create_endpoint(                                                      │
│   6034 │   │   │   endpoint_name=name,                                                           │
│   6035 │   │   │   config_name=name,                                                             │
│   6036 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:4867 in create_endpoint             │
│                                                                                                  │
│   4864 │   │   │   │   self.endpoint_arn = res["EndpointArn"]                                    │
│   4865 │   │   │                                                                                 │
│   4866 │   │   │   if wait:                                                                      │
│ ❱ 4867 │   │   │   │   self.wait_for_endpoint(endpoint_name, live_logging=live_logging)          │
│   4868 │   │   │   return endpoint_name                                                          │
│   4869 │   │   except Exception as e:                      

In [ ]:
import boto3
import json

runtime_client = boto3.client('sagemaker-runtime')

test_payload = {
    "event_count": 45,
    "page_view_count": 3,
    "unique_route_count": 2,
    "click_count": 12,
    "field_change_count": 5,
    "flow_success_count": 0,
    "flow_failure_count": 3,
    "error_event_count": 4,
    "retry_count": 2,
    "rage_click_count": 6,          # High frustration indicator
    "session_duration_ms": 150000,
    "total_dwell_ms": 80000,
    "avg_inter_event_gap_ms": 800
}

print(f"Sending payload to endpoint: frustration-ensemble-endpoint...")

response = runtime_client.invoke_endpoint(
    EndpointName="frustration-ensemble-endpoint",
    ContentType="application/json",
    Body=json.dumps(test_payload)
)

result = json.loads(response['Body'].read().decode())

print("\n--- API Response ---")
print(json.dumps(result, indent=2))

In [ ]:
sess = sagemaker.Session()

endpoint_name = "frustration-ensemble-endpoint"

print(f"Deleting endpoint: {endpoint_name}...")
sess.delete_endpoint(endpoint_name=endpoint_name)

print("Endpoint successfully deleted!")

In [4]:
%%writefile inference.py

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import argparse
import boto3
import json
import numpy as np
import pandas as pd

import preprocess as prep

# --- Constants & S3 Config ---
BUCKET_NAME = "sagemaker-studio-i0gutcxdy"
RESULTS_PREFIX = "frustration-model/results"
TRAIN_TEST_PREFIX = "frustration-model"
ENDPOINT_NAME = "frustration-ensemble-endpoint"

# Provided Thresholds (Anchored to a 9.0 Score)
AE_THRESH = 0.546781
IF_THRESH = 0.023487
GMM_THRESH = -8.274406

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def process_raw_data(raw_input_path, do_split_upload=False):
    print(f"\n--- 1. Processing Raw Telemetry Data ---")
    print(f"Loading raw telemetry from: {raw_input_path}")
    
    loader = prep.S3DataLoader(source=raw_input_path)
    raw_events = loader.fetchRawLogs()
    if not raw_events:
        raise RuntimeError("No telemetry events loaded.")

    aggregator = prep.SessionAggregator()
    aggregator.ingest_many(raw_events)
    sessions = aggregator.groupBySession()
    
    # Extract features but DO NOT normalize locally
    extractor = prep.FeatureExtractor(do_normalize=False)
    X_raw, feature_names, session_ids, session_timestamps = prep.build_feature_matrix(sessions, extractor)
    
    df_full = pd.DataFrame(X_raw, columns=feature_names)
    df_full['sessionId'] = session_ids
    df_full['timestamp'] = session_timestamps
    
    print(f"Successfully processed {len(df_full)} sessions.")
    
    if do_split_upload:
        print("\n--- Optional: Splitting Data and Uploading to S3 ---")
        train_data, test_data = np.split(df_full.sample(frac=1, random_state=42), [int(0.7 * len(df_full))])
        
        out_dir = './out'
        os.makedirs(out_dir, exist_ok=True)
        train_local = os.path.join(out_dir, "train.csv")
        test_local = os.path.join(out_dir, "test.csv")
        
        train_data.to_csv(train_local, index=False)
        test_data.to_csv(test_local, index=False)
        
        s3_client = boto3.client('s3')
        s3_client.upload_file(train_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/traindir/train.csv")
        s3_client.upload_file(test_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/testdir/test.csv")
        print(f"[OK] 70/30 Train/Test split uploaded to S3")
        
    return df_full, feature_names

def run_inference(args):
    df_full, feature_names = process_raw_data(args.input_raw, args.split_upload)
    
    session_ids = df_full['sessionId'].values
    timestamps = df_full['timestamp'].values
    features_df = df_full[feature_names] 
    
    print(f"\n--- 2. Fetching Raw Scores from SageMaker Endpoint ---")
    runtime_client = boto3.client('sagemaker-runtime')
    
    results = []
    total_sessions = len(features_df)

    for index, row in features_df.iterrows():
        response = runtime_client.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='application/json',
            Body=json.dumps(row.to_dict())
        )
        results.append(json.loads(response['Body'].read().decode()))
        
        if (index + 1) % 50 == 0:
            print(f"Processed {index + 1}/{total_sessions} sessions...")

    # Extract the raw batch scores from the endpoint's response
    ae_mse = np.array([r['breakdown']['ae_raw_mse'] for r in results])
    if_scores = np.array([r['breakdown']['if_raw_score'] for r in results])
    gmm_scores = np.array([r['breakdown']['gmm_raw_score'] for r in results])

    print("\n--- 3. Calculating 0-10 Severity Scores ---")
    
    # AE Scaling
    ae_min = np.min(ae_mse)
    ae_scale_factor = (AE_THRESH - ae_min) if (AE_THRESH - ae_min) > 0 else 1e-6
    ae_scores_0_to_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    # IF Scaling
    if_max = np.max(if_scores)
    if_scale_factor = (if_max - IF_THRESH) if (if_max - IF_THRESH) > 0 else 1e-6
    if_scores_0_to_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    # GMM Scaling
    gmm_min = np.min(gmm_scores)
    gmm_scale_factor = (GMM_THRESH - gmm_min) if (GMM_THRESH - gmm_min) > 0 else 1e-6
    gmm_scores_0_to_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    # Ensemble Scaling
    ensemble_scores_0_to_10 = (ae_scores_0_to_10 + if_scores_0_to_10 + gmm_scores_0_to_10) / 3.0

    print("Formatting output CSVs...")
    def build_df(scores):
        return pd.DataFrame({
            'sessionId': session_ids,
            'frustrationScore': np.round(scores, 2),
            'severity': [calculate_severity(s) for s in scores],
            'timestamp': timestamps
        })

    df_ae = build_df(ae_scores_0_to_10)
    df_if = build_df(if_scores_0_to_10)
    df_gmm = build_df(gmm_scores_0_to_10)
    df_ensemble = build_df(ensemble_scores_0_to_10)

    # 4. Save Locally
    out_dir = './inference_results'
    os.makedirs(out_dir, exist_ok=True)
    
    ae_out = os.path.join(out_dir, 'ae_predictions.csv')
    if_out = os.path.join(out_dir, 'if_predictions.csv')
    gmm_out = os.path.join(out_dir, 'gmm_predictions.csv')
    ens_out = os.path.join(out_dir, 'ensemble_predictions.csv')
    
    df_ae.to_csv(ae_out, index=False)
    df_if.to_csv(if_out, index=False)
    df_gmm.to_csv(gmm_out, index=False)
    df_ensemble.to_csv(ens_out, index=False)
    
    print(f"\nPredictions saved locally to {out_dir}/")
    print(f"Ensemble High Severity Count : {sum(df_ensemble['severity'] == 'High')}")

    # 5. Upload Results to S3
    print(f"\n--- 4. Uploading Results to S3 ---")
    s3_client = boto3.client('s3')
    files_to_upload = {
        ae_out: f"{RESULTS_PREFIX}/ae_predictions.csv",
        if_out: f"{RESULTS_PREFIX}/if_predictions.csv",
        gmm_out: f"{RESULTS_PREFIX}/gmm_predictions.csv",
        ens_out: f"{RESULTS_PREFIX}/ensemble_predictions.csv"
    }
    
    for local_file, s3_key in files_to_upload.items():
        s3_client.upload_file(local_file, BUCKET_NAME, s3_key)
        print(f"Uploaded -> s3://{BUCKET_NAME}/{s3_key}")
        
    print("\n[SUCCESS] End-to-end API Inference Complete.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-raw', type=str, required=True, help="Path to raw NDJSON/JSONL")
    parser.add_argument('--split-upload', action='store_true', help="Split 70/30 and upload train/test CSVs to S3")
    
    args = parser.parse_args()
    run_inference(args)

Writing inference.py


In [1]:
%%writefile requirements.txt
tensorflow>=2.13.0
scikit-learn==1.2.2
pyod==1.1.2
pandas>=2.0.0

Writing requirements.txt


In [2]:
%%writefile serve.py
import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from pyod.models.gmm import GMM

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def model_fn(model_dir):
    print("Loading Keras AutoEncoder...")
    ae_path = os.path.join(model_dir, 'autoencoder_model.keras')
    autoencoder = tf.keras.models.load_model(ae_path)
    
    print("Loading Isolation Forest...")
    iso_path = os.path.join(model_dir, 'isolation_forest.pkl')
    isolation_forest = joblib.load(iso_path)
    
    print("Loading GMM...")
    gmm_path = os.path.join(model_dir, 'gmm_model.pkl')
    gmm_model = joblib.load(gmm_path)
    
    print("Loading Scaler...")
    scaler_path = os.path.join(model_dir, 'scaler.pkl')
    scaler = joblib.load(scaler_path)
    
    print("Loading Metadata and Thresholds...")
    metadata_path = os.path.join(model_dir, 'model_metadata.joblib')
    metadata = joblib.load(metadata_path)
    
    return {
        'autoencoder': autoencoder,
        'isolation_forest': isolation_forest,
        'gmm_model': gmm_model,
        'scaler': scaler,
        'metadata': metadata
    }

def input_fn(request_body, request_content_type):
    if request_content_type == 'application/json':
        data = json.loads(request_body)
        return pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}. Expected 'application/json'.")

def predict_fn(input_data, model_dict):
    autoencoder = model_dict['autoencoder']
    isolation_forest = model_dict['isolation_forest']
    gmm_model = model_dict['gmm_model']
    scaler = model_dict['scaler']
    metadata = model_dict['metadata']
    
    ae_thresh = metadata['ae_threshold']
    iso_thresh = metadata['iso_threshold']
    gmm_thresh = metadata['gmm_threshold']
    
    features = input_data[metadata['feature_names']]
    X_scaled = scaler.transform(features).astype(np.float32)
    
    reconstructed = autoencoder.predict(X_scaled, verbose=0)
    ae_mse = np.mean(np.square(X_scaled - reconstructed), axis=1)
    if_scores = isolation_forest.decision_function(X_scaled)
    gmm_scores = gmm_model.decision_function(X_scaled)
    
    ae_min = metadata.get('ae_min_error', 0.0) 
    ae_scale_factor = (ae_thresh - ae_min) if (ae_thresh - ae_min) > 0 else 1e-6
    ae_score_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    if_max = 0.15 
    if_scale_factor = (if_max - iso_thresh) if (if_max - iso_thresh) > 0 else 1e-6
    if_score_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    gmm_min = np.min(gmm_scores) 
    gmm_scale_factor = (gmm_thresh - gmm_min) if (gmm_thresh - gmm_min) > 0 else 1e-6
    gmm_score_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    ensemble_score_10 = float((ae_score_10[0] + if_score_10[0] + gmm_score_10[0]) / 3.0)
    
    return {
        "frustrationScore": round(ensemble_score_10, 2),
        "severity": calculate_severity(ensemble_score_10),
        "breakdown": {
            "ae_raw_mse": float(ae_mse[0]),
            "if_raw_score": float(if_scores[0]),
            "gmm_raw_score": float(gmm_scores[0])
        }
    }

def output_fn(prediction, accept):
    if accept == 'application/json':
        return json.dumps(prediction), accept
    else:
        raise ValueError(f"Accept header must be 'application/json', got: {accept}")

Overwriting serve.py


In [3]:
sess = sagemaker.Session()
role = sagemaker.get_execution_role() 

current_dir = os.path.abspath(os.getcwd())
req_path = os.path.join(current_dir, "requirements.txt")
serve_path = os.path.join(current_dir, "serve.py") 

model_s3_path = "s3://sagemaker-studio-i0gutcxdy/frustration-model/models/model.tar.gz"

print(f"Using requirements from: {req_path}")
print(f"Using entry point from: {serve_path}")
print("Configuring the Ensemble Model for SageMaker...")

ensemble_model = SKLearnModel(
    model_data=model_s3_path,
    role=role,
    entry_point=serve_path,
    dependencies=[req_path],
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=sess
)
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=2048,
    max_concurrency=5
)

print("Deploying Serverless endpoint)")
predictor = ensemble_model.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name="frustration-serverless-endpoint"
)

print(f"\n[SUCCESS] Serverless Endpoint deployed at: {predictor.endpoint_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Using requirements from: /home/sagemaker-user/deployment/requirements.txt
Using entry point from: /home/sagemaker-user/deployment/serve.py
Configuring the Ensemble Model for SageMaker...
Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)
------------------------------------------

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:29                                                                                   │
│                                                                                                  │
│   26 )                                                                                           │
│   27                                                                                             │
│   28 print("Deploying endpoint... (This takes 3-5 minutes. Grab a coffee!)")                     │
│ ❱ 29 predictor = ensemble_model.deploy(                                                          │
│   30 │   initial_instance_count=1,                                                               │
│   31 │   instance_type="ml.m5.large",                                                            │
│   32 │   endpoint_name="frustration-ensemble-endpoint"                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model.py:1814 in deploy                        │
│                                                                                                  │
│   1811 │   │   │   │   )                                                                         │
│   1812 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1813 │   │   │   else:                                                                         │
│ ❱ 1814 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1815 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1816 │   │   │   │   │   production_variants=[production_variant],                             │
│   1817 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:6033 in                             │
│ endpoint_from_production_variants                                                                │
│                                                                                                  │
│   6030 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6031 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6032 │   │                                                                                     │
│ ❱ 6033 │   │   return self.create_endpoint(                                                      │
│   6034 │   │   │   endpoint_name=name,                                                           │
│   6035 │   │   │   config_name=name,                                                             │
│   6036 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/session.py:4867 in create_endpoint             │
│                                                                                                  │
│   4864 │   │   │   │   self.endpoint_arn = res["EndpointArn"]                                    │
│   4865 │   │   │                                                                                 │
│   4866 │   │   │   if wait:                                                                      │
│ ❱ 4867 │   │   │   │   self.wait_for_endpoint(endpoint_name, live_logging=live_logging)          │
│   4868 │   │   │   return endpoint_name                                                          │
│   4869 │   │   except Exception as e:                      

In [ ]:
import boto3
import json

runtime_client = boto3.client('sagemaker-runtime')

test_payload = {
    "event_count": 45,
    "page_view_count": 3,
    "unique_route_count": 2,
    "click_count": 12,
    "field_change_count": 5,
    "flow_success_count": 0,
    "flow_failure_count": 3,
    "error_event_count": 4,
    "retry_count": 2,
    "rage_click_count": 6,          # High frustration indicator
    "session_duration_ms": 150000,
    "total_dwell_ms": 80000,
    "avg_inter_event_gap_ms": 800
}

print(f"Sending payload to endpoint: frustration-ensemble-endpoint...")

response = runtime_client.invoke_endpoint(
    EndpointName="frustration-ensemble-endpoint",
    ContentType="application/json",
    Body=json.dumps(test_payload)
)

result = json.loads(response['Body'].read().decode())

print("\n--- API Response ---")
print(json.dumps(result, indent=2))

In [ ]:
sess = sagemaker.Session()

endpoint_name = "frustration-ensemble-endpoint"

print(f"Deleting endpoint: {endpoint_name}...")
sess.delete_endpoint(endpoint_name=endpoint_name)

print("Endpoint successfully deleted!")

In [4]:
%%writefile inference.py

#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import argparse
import boto3
import json
import numpy as np
import pandas as pd

import preprocess as prep

# --- Constants & S3 Config ---
BUCKET_NAME = "sagemaker-studio-i0gutcxdy"
RESULTS_PREFIX = "frustration-model/results"
TRAIN_TEST_PREFIX = "frustration-model"
ENDPOINT_NAME = "frustration-ensemble-endpoint"

# Provided Thresholds (Anchored to a 9.0 Score)
AE_THRESH = 0.546781
IF_THRESH = 0.023487
GMM_THRESH = -8.274406

def calculate_severity(score):
    if score >= 9.0:
        return 'High'
    elif score >= 7.0:
        return 'Medium'
    else:
        return 'Normal'

def process_raw_data(raw_input_path, do_split_upload=False):
    print(f"\n--- 1. Processing Raw Telemetry Data ---")
    print(f"Loading raw telemetry from: {raw_input_path}")
    
    loader = prep.S3DataLoader(source=raw_input_path)
    raw_events = loader.fetchRawLogs()
    if not raw_events:
        raise RuntimeError("No telemetry events loaded.")

    aggregator = prep.SessionAggregator()
    aggregator.ingest_many(raw_events)
    sessions = aggregator.groupBySession()
    
    # Extract features but DO NOT normalize locally
    extractor = prep.FeatureExtractor(do_normalize=False)
    X_raw, feature_names, session_ids, session_timestamps = prep.build_feature_matrix(sessions, extractor)
    
    df_full = pd.DataFrame(X_raw, columns=feature_names)
    df_full['sessionId'] = session_ids
    df_full['timestamp'] = session_timestamps
    
    print(f"Successfully processed {len(df_full)} sessions.")
    
    if do_split_upload:
        print("\n--- Optional: Splitting Data and Uploading to S3 ---")
        train_data, test_data = np.split(df_full.sample(frac=1, random_state=42), [int(0.7 * len(df_full))])
        
        out_dir = './out'
        os.makedirs(out_dir, exist_ok=True)
        train_local = os.path.join(out_dir, "train.csv")
        test_local = os.path.join(out_dir, "test.csv")
        
        train_data.to_csv(train_local, index=False)
        test_data.to_csv(test_local, index=False)
        
        s3_client = boto3.client('s3')
        s3_client.upload_file(train_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/traindir/train.csv")
        s3_client.upload_file(test_local, BUCKET_NAME, f"{TRAIN_TEST_PREFIX}/testdir/test.csv")
        print(f"[OK] 70/30 Train/Test split uploaded to S3")
        
    return df_full, feature_names

def run_inference(args):
    df_full, feature_names = process_raw_data(args.input_raw, args.split_upload)
    
    session_ids = df_full['sessionId'].values
    timestamps = df_full['timestamp'].values
    features_df = df_full[feature_names] 
    
    print(f"\n--- 2. Fetching Raw Scores from SageMaker Endpoint ---")
    runtime_client = boto3.client('sagemaker-runtime')
    
    results = []
    total_sessions = len(features_df)

    for index, row in features_df.iterrows():
        response = runtime_client.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType='application/json',
            Body=json.dumps(row.to_dict())
        )
        results.append(json.loads(response['Body'].read().decode()))
        
        if (index + 1) % 50 == 0:
            print(f"Processed {index + 1}/{total_sessions} sessions...")

    # Extract the raw batch scores from the endpoint's response
    ae_mse = np.array([r['breakdown']['ae_raw_mse'] for r in results])
    if_scores = np.array([r['breakdown']['if_raw_score'] for r in results])
    gmm_scores = np.array([r['breakdown']['gmm_raw_score'] for r in results])

    print("\n--- 3. Calculating 0-10 Severity Scores ---")
    
    # AE Scaling
    ae_min = np.min(ae_mse)
    ae_scale_factor = (AE_THRESH - ae_min) if (AE_THRESH - ae_min) > 0 else 1e-6
    ae_scores_0_to_10 = np.clip(9.0 * (ae_mse - ae_min) / ae_scale_factor, 0.0, 10.0)
    
    # IF Scaling
    if_max = np.max(if_scores)
    if_scale_factor = (if_max - IF_THRESH) if (if_max - IF_THRESH) > 0 else 1e-6
    if_scores_0_to_10 = np.clip(9.0 * (if_max - if_scores) / if_scale_factor, 0.0, 10.0)
    
    # GMM Scaling
    gmm_min = np.min(gmm_scores)
    gmm_scale_factor = (GMM_THRESH - gmm_min) if (GMM_THRESH - gmm_min) > 0 else 1e-6
    gmm_scores_0_to_10 = np.clip(9.0 * (gmm_scores - gmm_min) / gmm_scale_factor, 0.0, 10.0)
    
    # Ensemble Scaling
    ensemble_scores_0_to_10 = (ae_scores_0_to_10 + if_scores_0_to_10 + gmm_scores_0_to_10) / 3.0

    print("Formatting output CSVs...")
    def build_df(scores):
        return pd.DataFrame({
            'sessionId': session_ids,
            'frustrationScore': np.round(scores, 2),
            'severity': [calculate_severity(s) for s in scores],
            'timestamp': timestamps
        })

    df_ae = build_df(ae_scores_0_to_10)
    df_if = build_df(if_scores_0_to_10)
    df_gmm = build_df(gmm_scores_0_to_10)
    df_ensemble = build_df(ensemble_scores_0_to_10)

    # 4. Save Locally
    out_dir = './inference_results'
    os.makedirs(out_dir, exist_ok=True)
    
    ae_out = os.path.join(out_dir, 'ae_predictions.csv')
    if_out = os.path.join(out_dir, 'if_predictions.csv')
    gmm_out = os.path.join(out_dir, 'gmm_predictions.csv')
    ens_out = os.path.join(out_dir, 'ensemble_predictions.csv')
    
    df_ae.to_csv(ae_out, index=False)
    df_if.to_csv(if_out, index=False)
    df_gmm.to_csv(gmm_out, index=False)
    df_ensemble.to_csv(ens_out, index=False)
    
    print(f"\nPredictions saved locally to {out_dir}/")
    print(f"Ensemble High Severity Count : {sum(df_ensemble['severity'] == 'High')}")

    # 5. Upload Results to S3
    print(f"\n--- 4. Uploading Results to S3 ---")
    s3_client = boto3.client('s3')
    files_to_upload = {
        ae_out: f"{RESULTS_PREFIX}/ae_predictions.csv",
        if_out: f"{RESULTS_PREFIX}/if_predictions.csv",
        gmm_out: f"{RESULTS_PREFIX}/gmm_predictions.csv",
        ens_out: f"{RESULTS_PREFIX}/ensemble_predictions.csv"
    }
    
    for local_file, s3_key in files_to_upload.items():
        s3_client.upload_file(local_file, BUCKET_NAME, s3_key)
        print(f"Uploaded -> s3://{BUCKET_NAME}/{s3_key}")
        
    print("\n[SUCCESS] End-to-end API Inference Complete.")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('--input-raw', type=str, required=True, help="Path to raw NDJSON/JSONL")
    parser.add_argument('--split-upload', action='store_true', help="Split 70/30 and upload train/test CSVs to S3")
    
    args = parser.parse_args()
    run_inference(args)

Writing inference.py
